# HydroClaude 教程2：环状管网分析
# Tutorial 2: Looped Network Analysis

**作者 / Author**: HydroClaude Development Team  
**日期 / Date**: 2025-10-30  
**难度 / Level**: 中级 / Intermediate  
**时长 / Duration**: 25分钟

---

## 📚 教程目标 / Tutorial Objectives

在这个教程中，你将学会：
1. 创建环状管网拓扑
2. 理解Hardy Cross方法的原理
3. 分析环路流量分配
4. 多工况对比分析
5. 优化管网设计

---

## 🎯 问题描述 / Problem Description

我们将创建一个典型的城市供水环状管网：
- 1个水库（恒定水头）
- 6个用水点（Junction节点）
- 8根管道（形成2个环路）

```
         P1
R1 ─────→ J1 ─────→ J2
          │  ↖ P2    │
          │    ↖     │
       P4 │      ↖ P3│
          │   环1  ↖ │
          ↓    P6    ↓
          J4 ←───── J3
          │  P5      │
          │          │
       P7 │    环2 P8│
          │          │
          ↓          ↓
          J5 ←───── J6
              P9
```

---

## Step 1: 导入模块

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

sys.path.insert(0, os.path.dirname(os.getcwd()))

from network.pressure_pipe import create_pressure_pipe
from network.network_node import Junction, Reservoir
from network.network_topology import NetworkTopology
from solvers.hardy_cross_solver import HardyCrossSolver

print("✅ 模块导入成功！")

## Step 2: 创建环状管网

### 2.1 创建节点

In [ ]:
# 创建拓扑
topology = NetworkTopology("环状管网示例 / Looped Network Example")

# 水库
reservoir = Reservoir('R1', elevation=100, head=130)
topology.add_node(reservoir)
print(f"✅ 水库 R1: 标高={reservoir.elevation}m, 水头={reservoir.head}m")

# 用水节点（模拟不同的用水区域）
junction_data = [
    ('J1', 55, 8.0,  "商业区"),
    ('J2', 58, 6.0,  "住宅区A"),
    ('J3', 60, 10.0, "工业区"),
    ('J4', 57, 7.0,  "住宅区B"),
    ('J5', 56, 9.0,  "学校区"),
    ('J6', 59, 5.0,  "公园区"),
]

print("\n📍 用水节点:")
for jid, elev, demand_ls, desc in junction_data:
    junction = Junction(jid, elevation=elev, demand=demand_ls/1000)
    topology.add_node(junction)
    print(f"  {jid} ({desc}): 标高={elev}m, 需水量={demand_ls}L/s")

### 2.2 创建管道（形成环路）

In [ ]:
# 管道配置
pipe_configs = [
    # 主干线
    ('P1', 'R1', 'J1', 300, 600, 1.0, "主干线"),
    
    # 环路1
    ('P2', 'J1', 'J2', 200, 400, 0.5, "环路1"),
    ('P3', 'J2', 'J3', 200, 450, 0.5, "环路1"),
    ('P4', 'J1', 'J4', 200, 350, 0.5, "环路1"),
    ('P5', 'J4', 'J3', 200, 420, 0.5, "环路1"),
    
    # 环路2
    ('P6', 'J4', 'J5', 150, 380, 0.5, "环路2"),
    ('P7', 'J5', 'J6', 150, 350, 0.5, "环路2"),
    ('P8', 'J6', 'J3', 150, 400, 0.5, "环路2"),
]

print("\n🔗 管道连接:")
for pid, from_node, to_node, D_mm, L, K, desc in pipe_configs:
    pipe = create_pressure_pipe(pid, D_mm/1000, L, material='steel', K_minor=K)
    topology.add_pipe(pipe, from_node, to_node)
    print(f"  {pid} ({desc}): {from_node}→{to_node}, DN{D_mm}, L={L}m")

# 检测环路
loops = topology.find_loops()
print(f"\n🔄 检测到 {len(loops)} 个环路")
for i, loop in enumerate(loops, 1):
    print(f"  环路{i}: {' → '.join(loop)}")

## Step 3: Hardy Cross方法求解

### 3.1 基本原理

Hardy Cross方法是基于两个基本定律：
1. **节点流量守恒**: ∑Q_in = ∑Q_out + Q_demand
2. **环路水头平衡**: ∑ΔH = 0（沿环路一周水头变化为零）

迭代公式：
$$
\Delta Q = -\frac{\sum h_L}{n \sum \frac{h_L}{|Q|}}
$$

其中：
- ΔQ: 环路流量修正值
- h_L: 管道水头损失
- n: 指数（Darcy公式中n≈2）

In [ ]:
# 创建求解器
solver = HardyCrossSolver(
    topology=topology,
    max_iter=100,
    tol=1e-6,
    verbose=True
)

print("🔧 使用Hardy Cross方法求解...\n")

# 求解
flows, heads = solver.solve()

if solver.converged:
    print(f"\n✅ 求解收敛！")
    print(f"   迭代次数: {solver.iteration}")
    print(f"   最大残差: {solver.residual:.2e}")
else:
    print(f"\n❌ 求解未收敛")

## Step 4: 分析环路流量分配

### 4.1 查看每个环路的流量

In [ ]:
print("\n🔄 环路流量分析:")
print("=" * 70)

for i, loop in enumerate(loops, 1):
    print(f"\n环路 {i}: {' → '.join(loop)}")
    print("-" * 70)
    
    total_head_loss = 0.0
    
    # 获取环路中的管道
    loop_pipes = []
    for j in range(len(loop)):
        from_node = loop[j]
        to_node = loop[(j+1) % len(loop)]
        
        # 查找连接这两个节点的管道
        for pid, (fn, tn) in topology.connections.items():
            if (fn == from_node and tn == to_node) or (fn == to_node and tn == from_node):
                loop_pipes.append((pid, from_node, to_node))
                break
    
    # 分析每根管道
    for pid, fn, tn in loop_pipes:
        Q = flows[pid]
        pipe = topology.pipes[pid]
        
        # 计算水头损失
        h_L = pipe.head_loss(Q)
        
        # 根据流向调整符号
        if topology.connections[pid][0] == fn:
            total_head_loss += h_L
        else:
            total_head_loss -= h_L
        
        print(f"  {pid}: {fn}→{tn}, Q={Q*1000:+7.2f}L/s, h_L={h_L:+7.3f}m")
    
    print(f"  环路总水头损失: {total_head_loss:.6f}m (应≈0)")
    
    if abs(total_head_loss) < 0.01:
        print("  ✓ 环路水头平衡")
    else:
        print("  ⚠️ 环路水头不平衡")

print("\n" + "=" * 70)

### 4.2 节点压力分布

In [ ]:
print("\n📊 节点压力分布:")
print("=" * 80)
print(f"{'节点':<6} {'类型':<10} {'标高(m)':<10} {'水头(m)':<10} {'压力(m)':<10} {'状态'}")
print("=" * 80)

for nid, node in topology.nodes.items():
    head = heads[nid]
    pressure = head - node.elevation
    
    # 节点类型
    if isinstance(node, Reservoir):
        node_type = "水库"
        status = "-"
    elif isinstance(node, Junction):
        node_type = "用水点"
        if pressure >= 20:
            status = "✓ 良好"
        elif pressure >= 15:
            status = "⚠️ 偏低"
        else:
            status = "✗ 不足"
    
    print(f"{nid:<6} {node_type:<10} {node.elevation:<10.2f} {head:<10.2f} "
          f"{pressure:<10.2f} {status}")

print("=" * 80)

# 统计
junction_pressures = [heads[nid] - node.elevation 
                     for nid, node in topology.nodes.items() 
                     if isinstance(node, Junction)]

print(f"\n压力统计:")
print(f"  最小压力: {min(junction_pressures):.2f}m")
print(f"  最大压力: {max(junction_pressures):.2f}m")
print(f"  平均压力: {np.mean(junction_pressures):.2f}m")

## Step 5: 多工况分析

### 5.1 定义工况

In [ ]:
# 保存原始需水量
original_demands = {nid: node.demand for nid, node in topology.nodes.items()}

# 定义工况系数
scenarios = {
    '高峰工况': 1.8,   # 早晚高峰
    '平均工况': 1.0,   # 正常
    '低谷工况': 0.4,   # 夜间
}

results = {}

print("\n🔄 多工况分析:")
print("=" * 80)

for scenario_name, factor in scenarios.items():
    print(f"\n【{scenario_name}】 (系数={factor})")
    
    # 调整需水量
    for nid, original_demand in original_demands.items():
        topology.nodes[nid].demand = original_demand * factor
    
    # 求解
    solver = HardyCrossSolver(topology, max_iter=100, tol=1e-6, verbose=False)
    flows, heads = solver.solve()
    
    # 分析结果
    if solver.converged:
        junction_pressures = [heads[nid] - node.elevation 
                             for nid, node in topology.nodes.items() 
                             if isinstance(node, Junction)]
        
        velocities = []
        for pid, pipe in topology.pipes.items():
            Q = abs(flows[pid])
            A = np.pi * (pipe.D / 2) ** 2
            V = Q / A
            velocities.append(V)
        
        results[scenario_name] = {
            'min_pressure': min(junction_pressures),
            'max_pressure': max(junction_pressures),
            'avg_pressure': np.mean(junction_pressures),
            'max_velocity': max(velocities),
        }
        
        print(f"  ✓ 收敛 ({solver.iteration}次迭代)")
        print(f"  压力: {results[scenario_name]['min_pressure']:.2f} ~ "
              f"{results[scenario_name]['max_pressure']:.2f}m")
        print(f"  最大流速: {results[scenario_name]['max_velocity']:.2f}m/s")
    else:
        print(f"  ✗ 未收敛")

# 恢复原始需水量
for nid, original_demand in original_demands.items():
    topology.nodes[nid].demand = original_demand

print("\n" + "=" * 80)

### 5.2 工况对比可视化

In [ ]:
# 绘制对比图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 子图1：压力对比
scenario_names = list(results.keys())
categories = ['最小压力', '平均压力', '最大压力']

x = np.arange(len(categories))
width = 0.25
colors = ['#e74c3c', '#3498db', '#2ecc71']

for i, (scenario, color) in enumerate(zip(scenario_names, colors)):
    values = [
        results[scenario]['min_pressure'],
        results[scenario]['avg_pressure'],
        results[scenario]['max_pressure'],
    ]
    ax1.bar(x + i*width, values, width, label=scenario, color=color, alpha=0.8)

ax1.set_ylabel('Pressure (m)', fontsize=12, fontweight='bold')
ax1.set_title('Pressure Comparison Across Scenarios', fontsize=13, fontweight='bold')
ax1.set_xticks(x + width)
ax1.set_xticklabels(categories)
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')
ax1.axhline(y=15, color='r', linestyle='--', linewidth=1, alpha=0.7, label='Min (15m)')

# 子图2：流速对比
max_velocities = [results[s]['max_velocity'] for s in scenario_names]

ax2.bar(scenario_names, max_velocities, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Max Velocity (m/s)', fontsize=12, fontweight='bold')
ax2.set_title('Maximum Velocity Across Scenarios', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
ax2.axhline(y=3.0, color='r', linestyle='--', linewidth=2, label='Max Limit (3.0m/s)')
ax2.legend()

# 添加数值标签
for i, (scenario, v) in enumerate(zip(scenario_names, max_velocities)):
    ax2.text(i, v + 0.05, f'{v:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ 可视化完成！")

## 🎓 知识要点总结 / Key Takeaways

### 1. 环状管网的优势
- **供水可靠性高**: 一根管道故障时，可通过其他路径供水
- **压力分布均匀**: 多路径分配流量，降低水头损失
- **灵活性强**: 可适应需水量变化

### 2. Hardy Cross方法特点
- **适用范围**: 环状管网求解
- **收敛性**: 通常需要5-20次迭代
- **准确性**: 满足节点守恒和环路平衡

### 3. 多工况分析的重要性
- **高峰工况**: 检查是否有压力不足
- **低谷工况**: 检查是否有流速过低（沉积）
- **设计原则**: 按最不利工况设计

### 4. 工程优化方向
- **管径选择**: 平衡投资与水力性能
- **环路布置**: 合理布置环路提高可靠性
- **压力控制**: 高程差大时考虑分区供水

---

## 💡 练习建议 / Practice Suggestions

1. **修改管径**: 尝试调整某些管道的管径，观察对压力分布的影响
2. **增加环路**: 添加更多连接管道，形成更多环路
3. **故障分析**: 模拟某根管道关闭（设置很小的管径），看系统如何应对
4. **优化设计**: 在满足压力要求的前提下，最小化管道总投资

---

## 🚀 下一步学习 / Next Steps

继续学习以下教程：
1. **水锤分析** - 瞬态水力学
2. **管网优化** - 经济性分析
3. **水质模拟** - 污染物扩散

---

**祝学习愉快！/ Happy Learning!** 🎉